<a href="https://colab.research.google.com/github/visumania/DeepSexist/blob/main/DatasetManagement/Dataset_Task_3_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Montura de google colab para poder leer/escribir en Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Importación de librerías necesarias

In [13]:
import pandas as pd
import os
import cv2
from tqdm import tqdm

Tratamiento del fichero original para generar uno nuevo en donde solamente se recoja la información relevante para la tarea 3.3

Etiquetas a tratar:
- **IDEOLOGICAL AND INEQUALITY**: el texto desacredita el movimiento feminista, rechaza la desigualdad entre hombres y mujeres o presente a los hombres como víctimas de la opresión del género.
- **STEREOTYPING AND DOMINANCE**: el texto expresa ideas falsas sobre las mujeres que sugieren que son más adecuadas para desempeñar ciertos papeles (madre, esposa, cuidadora familiar, fiel, tierna, cariñosa, sumisa, etc.), o inadecuadas para ciertas tareas (conducir, trabajo duro, etc.), o afirmar que los hombres son de alguna manera superiores a las mujeres.
- **OBJECTIFICATION**: el texto presenta a las mujeres como objetos al margen de su dignidad y aspectos personales, o asume o describe determinadas cualidades físicas que deben tener las mujeres para cumplir los roles tradicionales de género (cumplimiento de cánones de belleza, hipersexualización de los atributos femeninos, cuerpos de las mujeres a disposición de los hombres, etc.).
- **SEXUAL VIOLENCE**: se hacen sugerencias sexuales, se piden favores sexuales o se ejerce el acoso de naturaleza sexual (violación o agresión sexual).
- **MISOGYNY AND NON-SEXUAL VIOLENCE**: el texto expresa odio y violencia hacia las mujeres.

Para esta tarea de clasificación multi-etiqueta, vamos a proponer trabajar con 1 fichero (extraído a partir del fichero original base proporcionado por la competición) el cuál abarca los conflictos de la siguiente manera:
- **EXIST2025_training_task3_3_ALL.csv**: el etiquetado final se hará siguiendo el comportamiento de la operación lógica *OR* con el etiquetado de cada uno de los anotadores


# Generación del fichero Base a partir del JSON original

## 1. Cargamos el JSON original

In [14]:
df_json = pd.read_json('/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training.json', orient='index')
df_json.reset_index(drop=True, inplace=True)

## 2. Cargamos el CSV maestro de la tarea 3.1



In [15]:
CSV_TRAIN_MASTER = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1.csv"
df_master = pd.read_csv(CSV_TRAIN_MASTER)

In [16]:
# Limpieza rápida del master por si tiene columnas residuales
if "Unnamed: 0" in df_master.columns:
    df_master.drop(columns=["Unnamed: 0"], inplace=True)
if "label_task_3_1_merged" in df_master.columns:
    df_master.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)

## 3. Fusión y filtrado

In [17]:
# Hacemos un INNER JOIN por 'id_EXIST'. Esto automáticamente descarta
# los vídeos con empate porque ya no existen en tu df_master.
df_fusion = pd.merge(df_master[['id_EXIST', 'label']], df_json, on='id_EXIST', how='inner')

# Nos quedamos SOLO con los vídeos misóginos (label == 1)
df_3_3 = df_fusion[df_fusion['label'] == 1].copy()

## 4. Procesamiento Tarea 3.3 (Multietiqueta)

In [18]:
# Función para aplanar la lista de listas (ej: [['OBJECTIFICATION'], ['-']] -> ['OBJECTIFICATION', '-'])
def aplanar_etiquetas(lista_de_listas):
    if not isinstance(lista_de_listas, list):
        return []
    etiquetas_planas = []
    for sublista in lista_de_listas:
        if isinstance(sublista, list):
            etiquetas_planas.extend(sublista)
        else:
            etiquetas_planas.append(sublista)
    return etiquetas_planas

df_3_3['todas_las_etiquetas'] = df_3_3['labels_task3_3'].apply(aplanar_etiquetas)

df_3_3.head()

,id_EXIST,label,id_Tiktok,lang,text,video,path_video,url,annotators,number_annotators,gender_annotators,labels_task3_1,labels_task3_2,labels_task3_3,split,todas_las_etiquetas
0,120001,1,7281385962049998086,es,cuando ves a las del 08 en la fiesta tu amigo...,7281385962049998086.mp4,videos/7281385962049998086.mp4,https://www.tiktok.com/@inazumamemes_/video/72...,"[Annotator_4, Annotator_8, Annotator_9]",3,"[F, M, M]","[YES, NO, YES]","[DIRECT, -, DIRECT]","[[OBJECTIFICATION], [-], [OBJECTIFICATION]]",TRAIN-VIDEO_ES,"[OBJECTIFICATION, -, OBJECTIFICATION]"
1,120002,1,7164058026352168197,es,mujer sola caminando por la calle | yo automat...,7164058026352168197.mp4,videos/7164058026352168197.mp4,https://www.tiktok.com/@humor_subvencionado/vi...,"[Annotator_4, Annotator_8]",2,"[F, M]","[YES, YES]","[DIRECT, DIRECT]","[[MISOGYNY-NON-SEXUAL-VIOLENCE], [MISOGYNY-NON...",TRAIN-VIDEO_ES,"[MISOGYNY-NON-SEXUAL-VIOLENCE, MISOGYNY-NON-SE..."
3,120004,1,7305803156074597665,es,confirman las chicas cogiendo confianza despué...,7305803156074597665.mp4,videos/7305803156074597665.mp4,https://www.tiktok.com/@joan_re_07/video/73058...,"[Annotator_3, Annotator_7, Annotator_9]",3,"[F, M, M]","[NO, YES, YES]","[-, DIRECT, DIRECT]","[[-], [STEREOTYPING-DOMINANCE], [STEREOTYPING-...",TRAIN-VIDEO_ES,"[-, STEREOTYPING-DOMINANCE, STEREOTYPING-DOMIN..."
4,120005,1,7318400739775204614,es,aplastar la realidad la gordita del salón tien...,7318400739775204614.mp4,videos/7318400739775204614.mp4,https://www.tiktok.com/@furbito_edits5/video/7...,"[Annotator_3, Annotator_7]",2,"[F, M]","[YES, YES]","[DIRECT, DIRECT]","[[STEREOTYPING-DOMINANCE], [STEREOTYPING-DOMIN...",TRAIN-VIDEO_ES,"[STEREOTYPING-DOMINANCE, STEREOTYPING-DOMINANCE]"
6,120007,1,7266886453198605574,es,"ti tok ""no soy machista, ayudo a lavar...",7266886453198605574.mp4,videos/7266886453198605574.mp4,https://www.tiktok.com/@josefx_skw23/video/726...,"[Annotator_3, Annotator_7, Annotator_9]",3,"[F, M, M]","[YES, YES, YES]","[DIRECT, JUDGEMENTAL, DIRECT]","[[IDEOLOGICAL-INEQUALITY, STEREOTYPING-DOMINAN...",TRAIN-VIDEO_ES,"[IDEOLOGICAL-INEQUALITY, STEREOTYPING-DOMINANC..."


In [19]:
etiquetas_objetivo = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE"
]

In [20]:
# Si al menos un anotador marcó la etiqueta, asignamos 1 (Unión de anotadores)
for etiqueta in etiquetas_objetivo:
    df_3_3[etiqueta] = df_3_3['todas_las_etiquetas'].apply(lambda x: 1 if etiqueta in x else 0)

## 5. Limpieza final y exportación

In [21]:
columnas_a_mantener = ["lang", "id_EXIST", "text", "path_video"] + etiquetas_objetivo
df_3_3_final = df_3_3[columnas_a_mantener]

ruta_salida = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_training_task3_3.csv'
df_3_3_final.to_csv(ruta_salida, index=False)

print(f"✅ Dataset Multietiqueta generado con éxito.")
print(f"Total de vídeos misóginos procesados: {len(df_3_3_final)}")

✅ Dataset Multietiqueta generado con éxito.
Total de vídeos misóginos procesados: 1202


In [22]:
print(df_3_3_final['IDEOLOGICAL-INEQUALITY'].value_counts())
print(df_3_3_final['MISOGYNY-NON-SEXUAL-VIOLENCE'].value_counts())
print(df_3_3_final['OBJECTIFICATION'].value_counts())
print(df_3_3_final['STEREOTYPING-DOMINANCE'].value_counts())
print(df_3_3_final['SEXUAL-VIOLENCE'].value_counts())

IDEOLOGICAL-INEQUALITY
0    687
1    515
Name: count, dtype: int64
MISOGYNY-NON-SEXUAL-VIOLENCE
0    1011
1     191
Name: count, dtype: int64
OBJECTIFICATION
0    904
1    298
Name: count, dtype: int64
STEREOTYPING-DOMINANCE
1    868
0    334
Name: count, dtype: int64
SEXUAL-VIOLENCE
0    1043
1     159
Name: count, dtype: int64


# Generacion de los ficheros Train/Test estáticos

In [23]:
!pip install scikit-multilearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 3.5 MB/s eta 0:00:00


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
from skmultilearn.model_selection import iterative_train_test_split

In [ ]:
# 1. Cargamos el dataset limpio que generamos anteriormente
RUTA_CSV = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_training_task3_3.csv'
df_3_3 = pd.read_csv(RUTA_CSV)

etiquetas_objetivo = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE"
]

In [ ]:
# 2. Separar características (X) y etiquetas (y) en formato Numpy
# X contendrá todas las columnas excepto las 5 etiquetas
X = df_3_3.drop(columns=etiquetas_objetivo).values
# y contendrá solo las 5 columnas de etiquetas
y = df_3_3[etiquetas_objetivo].values

In [ ]:
# 3. Realizar el split estratificado multietiqueta (80% train, 20% test)
X_train, y_train, X_test, y_test = iterative_train_test_split(X, y, test_size=0.2)

In [ ]:
# 4. Reconstruir los DataFrames
columnas_X = df_3_3.columns.drop(etiquetas_objetivo)

df_train = pd.DataFrame(X_train, columns=columnas_X)
df_train_labels = pd.DataFrame(y_train, columns=etiquetas_objetivo)
train_df = pd.concat([df_train, df_train_labels], axis=1)

df_test = pd.DataFrame(X_test, columns=columnas_X)
df_test_labels = pd.DataFrame(y_test, columns=etiquetas_objetivo)
test_df = pd.concat([df_test, df_test_labels], axis=1)

In [ ]:
# 5. Guardar los ficheros
RUTA_TRAIN = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_train_3_3.csv'
RUTA_TEST = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_test_3_3.csv'

train_df.to_csv(RUTA_TRAIN, index=False)
test_df.to_csv(RUTA_TEST, index=False)

print("✅ Ficheros Train y Test generados correctamente.")
print(f"Tamaño de Train: {len(train_df)} vídeos")
print(f"Tamaño de Test: {len(test_df)} vídeos\n")

In [ ]:
# Comprobación de proporciones
print("Distribución IDEOLOGICAL-INEQUALITY en Train:")
print(train_df['IDEOLOGICAL-INEQUALITY'].value_counts())
print("Distribución IDEOLOGICAL-INEQUALITY en Test:")
print(test_df['IDEOLOGICAL-INEQUALITY'].value_counts())

print("Distribución STEREOTYPING-DOMINANCE en Train:")
print(train_df['STEREOTYPING-DOMINANCE'].value_counts())
print("Distribución STEREOTYPING-DOMINANCE en Test:")
print(test_df['STEREOTYPING-DOMINANCE'].value_counts())

print("Distribución OBJECTIFICATION en Train:")
print(train_df['OBJECTIFICATION'].value_counts())
print("Distribución OBJECTIFICATION en Test:")
print(test_df['OBJECTIFICATION'].value_counts())

print("Distribución SEXUAL-VIOLENCE en Train:")
print(train_df['SEXUAL-VIOLENCE'].value_counts())
print("Distribución SEXUAL-VIOLENCE en Test:")
print(test_df['SEXUAL-VIOLENCE'].value_counts())

print("Distribución MISOGYNY-NON-SEXUAL-VIOLENCE en Train:")
print(train_df['MISOGYNY-NON-SEXUAL-VIOLENCE'].value_counts())
print("Distribución MISOGYNY-NON-SEXUAL-VIOLENCE en Test:")
print(test_df['MISOGYNY-NON-SEXUAL-VIOLENCE'].value_counts())

In [ ]:
# Proporciones de las distribuciones
print("Distribución porcentual IDEOLOGICAL-INEQUALITY en Train:")
print(train_df['IDEOLOGICAL-INEQUALITY'].value_counts(normalize=True))
print("Distribución porcentual IDEOLOGICAL-INEQUALITY en Test:")
print(test_df['IDEOLOGICAL-INEQUALITY'].value_counts(normalize=True))

print("Distribución porcentual STEREOTYPING-DOMINANCE en Train:")
print(train_df['STEREOTYPING-DOMINANCE'].value_counts(normalize=True))
print("Distribución porcentual STEREOTYPING-DOMINANCE en Test:")
print(test_df['STEREOTYPING-DOMINANCE'].value_counts(normalize=True))

print("Distribución porcentual OBJECTIFICATION en Train:")
print(train_df['OBJECTIFICATION'].value_counts(normalize=True))
print("Distribución porcentual OBJECTIFICATION en Test:")
print(test_df['OBJECTIFICATION'].value_counts(normalize=True))

print("Distribución porcentual SEXUAL-VIOLENCE en Train:")
print(train_df['SEXUAL-VIOLENCE'].value_counts(normalize=True))
print("Distribución porcentual SEXUAL-VIOLENCE en Test:")
print(test_df['SEXUAL-VIOLENCE'].value_counts())

print("Distribución porcentual MISOGYNY-NON-SEXUAL-VIOLENCE en Train:")
print(train_df['MISOGYNY-NON-SEXUAL-VIOLENCE'].value_counts(normalize=True))
print("Distribución porcentual MISOGYNY-NON-SEXUAL-VIOLENCE en Test:")
print(test_df['MISOGYNY-NON-SEXUAL-VIOLENCE'].value_counts(normalize=True))

# Generación de dataset de frames para entrenamiento con modelo de imagen

## 1. Configuración

In [24]:
NUM_FOTOGRAMAS_POR_VIDEO = 4  # O los frames que prefieras (ej. 4 u 8)
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/"

In [25]:
# Vamos a hacerlo para el conjunto de Train
CARPETA_DESTINO_IMAGENES_TRAIN = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Frames/Frames-{NUM_FOTOGRAMAS_POR_VIDEO}/Train"
os.makedirs(CARPETA_DESTINO_IMAGENES_TRAIN, exist_ok=True)

# Vamos a hacerlo para el conjunto de Test
CARPETA_DESTINO_IMAGENES_TEST = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Frames/Frames-{NUM_FOTOGRAMAS_POR_VIDEO}/Test"
os.makedirs(CARPETA_DESTINO_IMAGENES_TEST, exist_ok=True)

In [26]:
# Ruta del CSV de Train de la Tarea 3.3 que generamos antes (el que ya tiene las 5 etiquetas)
CSV_ORIGINAL_TRAIN = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_train_3_3.csv"
CSV_NUEVO_IMAGENES_TRAIN = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Frames/Frames-{NUM_FOTOGRAMAS_POR_VIDEO}/dataset_imagenes_train_multietiqueta.csv"

# Ruta del CSV de Test de la Tarea 3.3 que generamos antes (el que ya tiene las 5 etiquetas)
CSV_ORIGINAL_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_test_3_3.csv"
CSV_NUEVO_IMAGENES_TEST = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Frames/Frames-{NUM_FOTOGRAMAS_POR_VIDEO}/dataset_imagenes_test_multietiqueta.csv"

## 2. Carga del dataframe

In [27]:
df_train = pd.read_csv(CSV_ORIGINAL_TRAIN)
print(f"Cargados {len(df_train)} vídeos de entrenamiento para extracción de frames.")

df_test = pd.read_csv(CSV_ORIGINAL_TEST)
print(f"Cargados {len(df_test)} vídeos de test para extracción de frames.")

etiquetas_objetivo = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE"
]

Cargados 972 vídeos de entrenamiento para extracción de frames.
Cargados 230 vídeos de test para extracción de frames.


## 3. Bucle de extracción (adaptado a multietiqueta)

In [28]:
# Extracción de Train

datos_imagenes_train = []
print(f"Extrayendo {NUM_FOTOGRAMAS_POR_VIDEO} fotogramas por vídeo...")

for index, row in tqdm(df_train.iterrows(), total=len(df_train)):
    id_video = row['id_EXIST']
    ruta_relativa_video = row['path_video']

    # Recogemos todas las etiquetas multietiqueta de este vídeo
    etiquetas_row = {etiq: row[etiq] for etiq in etiquetas_objetivo}

    # Construimos la ruta absoluta del .mp4
    ruta_absoluta = os.path.join(RUTA_BASE_VIDEOS, ruta_relativa_video)

    if not os.path.exists(ruta_absoluta):
        continue  # Si falta algún vídeo, nos lo saltamos

    video = cv2.VideoCapture(ruta_absoluta)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        video.release()
        continue

    # Calculamos los índices de los fotogramas de forma uniforme
    step = total_frames // (NUM_FOTOGRAMAS_POR_VIDEO + 1)
    frames_indices = [step * i for i in range(1, NUM_FOTOGRAMAS_POR_VIDEO + 1)]

    for idx, frame_idx in enumerate(frames_indices):
        video.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = video.read()

        if ret:
            nombre_imagen = f"{id_video}_frame_{idx}.jpg"
            ruta_guardado = os.path.join(CARPETA_DESTINO_IMAGENES_TRAIN, nombre_imagen)

            cv2.imwrite(ruta_guardado, frame)

            # Creamos el diccionario base con la imagen y el ID
            info_frame = {
                'id_EXIST': id_video,
                'path_imagen': ruta_guardado,
                'lang': row['lang'],
                'text': row['text']
            }
            # Añadimos dinámicamente las 5 etiquetas multietiqueta (heredadas del vídeo padre)
            info_frame.update(etiquetas_row)

            datos_imagenes_train.append(info_frame)

    video.release()

Extrayendo 4 fotogramas por vídeo...


100%|██████████| 972/972 [26:07<00:00,  1.61s/it]


In [29]:
# Extracción de Test

datos_imagenes_test = []
print(f"Extrayendo {NUM_FOTOGRAMAS_POR_VIDEO} fotogramas por vídeo...")

for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    id_video = row['id_EXIST']
    ruta_relativa_video = row['path_video']

    # Recogemos todas las etiquetas multietiqueta de este vídeo
    etiquetas_row = {etiq: row[etiq] for etiq in etiquetas_objetivo}

    # Construimos la ruta absoluta del .mp4
    ruta_absoluta = os.path.join(RUTA_BASE_VIDEOS, ruta_relativa_video)

    if not os.path.exists(ruta_absoluta):
        continue  # Si falta algún vídeo, nos lo saltamos

    video = cv2.VideoCapture(ruta_absoluta)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        video.release()
        continue

    # Calculamos los índices de los fotogramas de forma uniforme
    step = total_frames // (NUM_FOTOGRAMAS_POR_VIDEO + 1)
    frames_indices = [step * i for i in range(1, NUM_FOTOGRAMAS_POR_VIDEO + 1)]

    for idx, frame_idx in enumerate(frames_indices):
        video.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = video.read()

        if ret:
            nombre_imagen = f"{id_video}_frame_{idx}.jpg"
            ruta_guardado = os.path.join(CARPETA_DESTINO_IMAGENES_TEST, nombre_imagen)

            cv2.imwrite(ruta_guardado, frame)

            # Creamos el diccionario base con la imagen y el ID
            info_frame = {
                'id_EXIST': id_video,
                'path_imagen': ruta_guardado,
                'lang': row['lang'],
                'text': row['text']
            }
            # Añadimos dinámicamente las 5 etiquetas multietiqueta (heredadas del vídeo padre)
            info_frame.update(etiquetas_row)

            datos_imagenes_test.append(info_frame)

    video.release()

Extrayendo 4 fotogramas por vídeo...


100%|██████████| 230/230 [06:06<00:00,  1.59s/it]


## 4. Guardar los nuevos dataset de imágenes

In [30]:
df_imagenes_train = pd.DataFrame(datos_imagenes_train)
df_imagenes_train.to_csv(CSV_NUEVO_IMAGENES_TRAIN, index=False)

print(f"\n¡Extracción completada! Se han guardado {len(df_imagenes_train)} imágenes.")
print(f"Nuevo CSV generado en: {CSV_NUEVO_IMAGENES_TRAIN}")

df_imagenes_test = pd.DataFrame(datos_imagenes_test)
df_imagenes_test.to_csv(CSV_NUEVO_IMAGENES_TEST, index=False)

print(f"\n¡Extracción completada! Se han guardado {len(df_imagenes_test)} imágenes.")
print(f"Nuevo CSV generado en: {CSV_NUEVO_IMAGENES_TEST}")


¡Extracción completada! Se han guardado 3888 imágenes.
Nuevo CSV generado en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Frames/Frames-4/dataset_imagenes_train_multietiqueta.csv

¡Extracción completada! Se han guardado 920 imágenes.
Nuevo CSV generado en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Frames/Frames-4/dataset_imagenes_test_multietiqueta.csv
